# Moral foundation coding with an OpenAI model — main analysis

The baseline cell: **main prompt × 0-shot**, model `gpt-6-luna` (non-reasoning, `reasoning_effort="none"`).

Before running:
* Select the Python kernel (top right) you want to use.
* Provide your OpenAI API key (prompted below, or set `OPENAI_API_KEY` in a `.env` file — keep `.env` out of git).

## Setup

In [ ]:
# Run once, then restart the kernel
%pip install -q -U openai pandas tqdm scikit-learn ipywidgets python-dotenv

In [1]:
# Reload run_grid_openai.py automatically whenever you edit it
%load_ext autoreload
%autoreload 2

In [2]:
import os, json, getpass
import pandas as pd
from sklearn.metrics import precision_recall_fscore_support, cohen_kappa_score

try:
    from dotenv import load_dotenv
    load_dotenv()                      # reads OPENAI_API_KEY from .env if present
except ImportError:
    pass
if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key: ")

import run_grid_openai as rg
print("model:", rg.MODEL, "| reasoning_effort:", rg.REASONING_EFFORT, "| output folder:", rg.OUT_DIR)

model: gpt-6-luna | reasoning_effort: none | output folder: runs/gpt-6-luna


## Load validation set

In [3]:
validation = rg.load_validation()
print(validation.shape)
validation.head()

(50, 20)


,para_id,moral_Yunfan,care_Yunfan,fairness_Yunfan,loyalty_Yunfan,authority_Yunfan,sanctity_Yunfan,moral_Wiktoria,care_Wiktoria,fairness_Wiktoria,loyalty_Wiktoria,authority_Wiktoria,sanctity_Wiktoria,moral_Carlos,care_Carlos,fairness_Carlos,loyalty_Carlos,authority_Carlos,sanctity_Carlos,text
0,CREC-2025-06-12-pt1-PgH2814_2_3,1,1,0,0,0,1,1,1,0,1,1,0,1,1,0,0,1,0,Every American should be outraged at how the T...
1,CREC-2025-02-07-pt1-PgH576-2_1_26,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,How can this not terrify people? Are there peo...
2,CREC-2025-06-05-pt1-PgH2489_13_1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,I rise today in strong opposition to H.R. 2987...
3,CREC-2025-02-25-pt1-PgH791-3_73_2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,This budget will usher in a new golden era tha...
4,CREC-2026-02-25-pt1-PgH2301-2_16_9,1,0,0,0,1,0,1,0,0,0,1,0,1,1,0,0,1,0,Even after the United States supreme_court rul...


## Prompt
The main prompt is identical to the one in `llm_moral_coding.ipynb`.

In [7]:
print(rg.make_prompt(validation["text"][0]))



1. TASK
You're an expert political scientist analyzing U.S. congressional floor speeches. You need to decide whether a paragraph from the speeches uses moral rhetoric, and if it does, which moral foundations it draws on.
Code six labels, each as 1 (present) or 0 (absent): MORAL RHETORIC, CARE, FAIRNESS, LOYALTY, AUTHORITY, SANCTITY.

2. CODING APPROACH
* The unit of analysis is the paragraph. Read the full paragraph before coding.
* Code each label for presence (1) or absence (0).
* Code a label as 1 if the paragraph expresses its meaning at least once.
* A paragraph can be coded for several moral foundations at the same time.

Procedure:
1. Read the full paragraph.
2. Code MORAL RHETORIC as 1 or 0.
3. If MORAL RHETORIC is 1, code each of the five moral foundations as 1 or 0. At least one foundation must be 1.
4. If MORAL RHETORIC is 0, all five moral foundations must be 0.

3. LABELS


Label: MORAL RHETORIC
Definition: The speaker, explicitly or implicitly,
* judges an action, actor

Test on one paragraph:

In [8]:
rg.check_params(rg.make_prompt(validation["text"][0]))   # drops temperature/seed if the model rejects them
rg.ask_llm(rg.make_prompt(validation["text"][0]), validation["text"][0])

{'moral_rhetoric': 1,
 'care': 1,
 'fairness': 0,
 'loyalty': 1,
 'authority': 1,
 'sanctity': 0,
 'evidence_moral_rhetoric': '“outraged” at the administration’s “horrible actions”',
 'evidence_care': '“ensure that my constituents, immigrants, and all Americans are protected”',
 'evidence_fairness': '',
 'evidence_loyalty': '“Every American” and “all Americans”',
 'evidence_authority': '“It is my job to ensure” that people are protected',
 'evidence_sanctity': '',
 'problem': None,
 'warnings': ['moral_rhetoric: evidence not found in paragraph',
  'loyalty: evidence not found in paragraph',
  'authority: evidence not found in paragraph'],
 'raw': '{"evidence":{"moral_rhetoric":"“outraged” at the administration’s “horrible actions”","care":"“ensure that my constituents, immigrants, and all Americans are protected”","fairness":"","loyalty":"“Every American” and “all Americans”","authority":"“It is my job to ensure” that people are protected","sanctity":""},"moral_rhetoric":1,"care":1,"fa

## Run the primary cell (main × 0-shot)

Every paragraph is coded once; no resume logic. Results are saved to
`runs/gpt-6-luna/main_0shot_validation.csv`, together with the exact prompt (`_prompt.txt`)
and the settings used (`_settings.json`).

In [10]:
pred = rg.run_cell(validation, "main", 0)
pred.head()

main, 0-shot:   0%|          | 0/50 [00:00<?, ?it/s]

,para_id,text,variant,shots,moral_rhetoric,care,fairness,loyalty,authority,sanctity,evidence_moral_rhetoric,evidence_care,evidence_fairness,evidence_loyalty,evidence_authority,evidence_sanctity,problem,warnings,raw,model_version
0,CREC-2025-06-12-pt1-PgH2814_2_3,Every American should be outraged at how the T...,main,0,1,1,1,0,1,0,“outraged” at the administration’s “horrible a...,"“ensure that my constituents, immigrants, and ...","“Whether you are a U.S. Senator, a U.S. Citize...",,“It is my job to ensure” that people “are prot...,,None,[moral_rhetoric: evidence not found in paragra...,"{""evidence"":{""moral_rhetoric"":""“outraged” at t...",gpt-6-luna
1,CREC-2025-02-07-pt1-PgH576-2_1_26,How can this not terrify people? Are there peo...,main,0,0,0,0,0,0,0,,,,,,,None,[],"{""evidence"":{""moral_rhetoric"":"""",""care"":"""",""fa...",gpt-6-luna
2,CREC-2025-06-05-pt1-PgH2489_13_1,I rise today in strong opposition to H.R. 2987...,main,0,0,0,0,0,0,0,,,,,,,None,[],"{""evidence"":{""moral_rhetoric"":"""",""care"":"""",""fa...",gpt-6-luna
3,CREC-2025-02-25-pt1-PgH791-3_73_2,This budget will usher in a new golden era tha...,main,0,0,0,0,0,0,0,,,,,,,None,[],"{""evidence"":{""moral_rhetoric"":"""",""care"":"""",""fa...",gpt-6-luna
4,CREC-2026-02-25-pt1-PgH2301-2_16_9,Even after the United States supreme_court rul...,main,0,1,1,0,0,1,0,“the President doesn't have the authority to d...,"“on households, making their groceries, their ...",,,“the President doesn't have the authority to d...,,None,[],"{""evidence"":{""moral_rhetoric"":""“the President ...",gpt-6-luna


In [11]:
settings = json.load(open(f"{rg.OUT_DIR}/main_0shot_validation_settings.json"))
print("dropped parameters:", settings["dropped_params"])     # [] = temperature and seed were used
print("model versions returned:", settings["model_versions"])
print("params:", settings["params"])

dropped parameters: []
model versions returned: ['gpt-6-luna']
params: {'model': 'gpt-6-luna', 'temperature': 0, 'seed': 502, 'reasoning_effort': 'none'}


## Output checks

In [12]:
print("problems:")
print(pred["problem"].fillna("none").value_counts())
warnings = pred["warnings"].apply(lambda w: w if isinstance(w, list) else [])
print("\nwarnings:")
print(pd.Series([x.split(": ", 1)[1] for ws in warnings for x in ws], dtype=str).value_counts())

problems:
problem
none                        48
moral_without_foundation     1
foundation_without_moral     1
Name: count, dtype: int64

warnings:
evidence not found in paragraph    9
coded 1 without evidence           2
evidence given but coded 0         1
Name: count, dtype: int64


## Evaluation against human labels

In [13]:
EVAL_LABELS = ["moral"] + rg.FOUNDATIONS
CODERS = ["Yunfan", "Wiktoria", "Carlos"]
LLM_COL = {"moral": "moral_rhetoric", **{f: f for f in rg.FOUNDATIONS}}

m = validation.merge(pred.drop(columns=["text"], errors="ignore"), on="para_id")
m = m[m["moral_rhetoric"].notna()]          # drop parse/API errors
for l in EVAL_LABELS:                        # majority vote of the three coders
    m[f"{l}_majority"] = (m[[f"{l}_{c}" for c in CODERS]].sum(axis=1) >= 2).astype(int)

def evaluate(df, human_col_fn, llm_col_map=LLM_COL):
    rows = []
    for l in EVAL_LABELS:
        sub = df[[human_col_fn(l), llm_col_map[l]]].dropna()
        y, yhat = sub.iloc[:, 0].astype(int), sub.iloc[:, 1].astype(int)
        p, r, f, _ = precision_recall_fscore_support(y, yhat, average="binary", zero_division=0)
        rows.append({"label": l, "n": len(sub), "human_n": y.sum(), "llm_n": yhat.sum(),
                     "accuracy": (y == yhat).mean(), "precision": p, "recall": r, "f1": f,
                     "kappa": cohen_kappa_score(y, yhat)})
    return pd.DataFrame(rows).set_index("label").round(3)

all_results = {c: evaluate(m, lambda l, c=c: f"{l}_{c}") for c in CODERS}
all_results["majority"] = evaluate(m, lambda l: f"{l}_majority")
results = pd.concat(all_results, names=["coder", "label"])

print("n =", len(m))
for c, res in all_results.items():
    print(f"{c:>9} macro-F1 (5 foundations):", res.loc[rg.FOUNDATIONS, "f1"].mean().round(3))
results

n = 50
   Yunfan macro-F1 (5 foundations): 0.503
 Wiktoria macro-F1 (5 foundations): 0.313
   Carlos macro-F1 (5 foundations): 0.623
 majority macro-F1 (5 foundations): 0.594


n  human_n  llm_n  accuracy  precision  recall     f1  \
coder    label                                                               
Yunfan   moral      50       19     23      0.80      0.696   0.842  0.762   
         care       50        5     12      0.82      0.333   0.800  0.471   
         fairness   50        4      4      0.92      0.500   0.500  0.500   
         loyalty    50        4     12      0.80      0.250   0.750  0.375   
         authority  50        5     13      0.84      0.385   1.000  0.556   
         sanctity   50        7      6      0.90      0.667   0.571  0.615   
Wiktoria moral      50       21     23      0.68      0.609   0.667  0.636   
         care       50       11     12      0.74      0.417   0.455  0.435   
         fairness   50        3      4      0.86      0.000   0.000  0.000   
         loyalty    50        4     12      0.76      0.167   0.500  0.250   
         authority  50        8     13      0.74      0.308   0.500  0.381   
         sanctity   50        2      6      0.92      0.333   1.000  0.500   
Carlos   moral      50       27     23      0.80      0.870   0.741  0.800   
         care       50        9     12      0.90      0.667   0.889  0.762   
         fairness   50        6      4      0.88      0.500   0.333  0.400   
         loyalty    50        5     12      0.78      0.250   0.600  0.353   
         authority  50       12     13      0.90      0.769   0.833  0.800   
         sanctity   50        4      6      0.96      0.667   1.000  0.800   
majority moral      50       24     23      0.82      0.826   0.792  0.809   
         care       50        7     12      0.82      0.417   0.714  0.526   
         fairness   50        3      4      0.94      0.500   0.667  0.571   
         loyalty    50        5     12      0.82      0.333   0.800  0.471   
         authority  50        7     13      0.84      0.462   0.857  0.600   
         sanctity   50        4      6      0.96      0.667   1.000  0.800   

                    kappa  
coder    label             
Yunfan   moral      0.592  
         care       0.384  
         fairness   0.457  
         loyalty    0.290  
         authority  0.481  
         sanctity   0.558  
Wiktoria moral      0.352  
         care       0.266  
         fairness  -0.074  
         loyalty    0.148  
         authority  0.228  
         sanctity   0.468  
Carlos   moral      0.603  
         care       0.700  
         fairness   0.336  
         loyalty    0.247  
         authority  0.733  
         sanctity   0.779  
majority moral      0.639  
         care       0.425  
         fairness   0.540  
         loyalty    0.384  
         authority  0.511  
         sanctity   0.779

In [15]:
results.to_csv(f"{rg.OUT_DIR}/main_0shot_evaluation.csv")

### LLM–human agreement vs. human–human agreement (Cohen's κ)

Is the LLM within the range of agreement among the human coders?

In [14]:
from itertools import combinations

rows = []
for l in EVAL_LABELS:
    hh = [cohen_kappa_score(m[f"{l}_{a}"].astype(int), m[f"{l}_{b}"].astype(int))
          for a, b in combinations(CODERS, 2)]
    lh = [cohen_kappa_score(m[f"{l}_{c}"].astype(int), m[LLM_COL[l]].astype(int)) for c in CODERS]
    rows.append({"label": l,
                 "human-human mean": sum(hh) / 3, "human-human min": min(hh), "human-human max": max(hh),
                 "LLM-human mean": sum(lh) / 3, "LLM-human min": min(lh), "LLM-human max": max(lh)})
pd.DataFrame(rows).set_index("label").round(3)

,human-human mean,human-human min,human-human max,LLM-human mean,LLM-human min,LLM-human max
label,,,,,,
moral,0.438,0.368,0.529,0.515,0.352,0.603
care,0.345,0.252,0.508,0.450,0.266,0.700
fairness,0.132,-0.087,0.558,0.240,-0.074,0.457
loyalty,0.322,0.146,0.634,0.228,0.148,0.290
authority,0.429,0.381,0.521,0.481,0.228,0.733
sanctity,0.584,0.408,0.696,0.602,0.468,0.779


## Disagreements for error analysis

In [ ]:
EVID_COL = {l: f"evidence_{LLM_COL[l]}" for l in EVAL_LABELS}

def find_disagreements(df, coder="majority", outfile=None):
    """coder: 'Yunfan' / 'Wiktoria' / 'Carlos' / 'majority'"""
    def diff(row):
        out = []
        for l in EVAL_LABELS:
            h, p = row[f"{l}_{coder}"], row[LLM_COL[l]]
            if pd.isna(h) or pd.isna(p):
                continue
            if int(h) != int(p):
                out.append(l)
        return out

    df = df.copy()
    df["disagree_on"] = df.apply(diff, axis=1)
    errors = df[df["disagree_on"].str.len() > 0]
    cols = (["para_id", "text", "disagree_on"]
            + [c for l in EVAL_LABELS for c in (f"{l}_{coder}", LLM_COL[l])]
            + [EVID_COL[l] for l in EVAL_LABELS])
    if outfile:
        errors[cols].to_csv(outfile, index=False)
    print(f"{coder}: {len(errors)} / {len(df)} paragraphs with at least one disagreement")
    return errors[cols]

errors = find_disagreements(m, coder="majority",
                            outfile=f"{rg.OUT_DIR}/main_0shot_disagreements_majority.csv")
errors

# Full grid: 3 prompt variants × 3 shot conditions

Needs, in the same folder:
* `fewshot_examples.csv` — the few-shot examples in order (first 2 rows = 2-shot, first 5 = 5-shot).
* `DEFINITIONS_PARAPHRASE` in `run_grid_openai.py` — reviewed paraphrased definitions.

Example paragraphs that are also in the validation set are left out of the 2- and 5-shot cells,
so all cells are compared on the paragraphs they have in common.

In [ ]:
examples = rg.load_examples()
examples[["para_id"] + rg.LABELS]
print(rg.make_examples(examples.head(2)))          # check the 2-shot block

In [ ]:
# The primary cell (main x 0-shot) was already run above as `pred`, so run the other 8 cells.
# Each is saved as runs/gpt-6-luna/{variant}_{shots}shot_validation.csv
other_cells = [(v, k) for v in rg.VARIANTS for k in rg.SHOTS if (v, k) != rg.PRIMARY]
all_pred = pd.concat([pred, rg.run_grid(validation, cells=other_cells, examples=examples)],
                     ignore_index=True)
all_pred.groupby(["variant", "shots"]).size()

## Robustness cells (run later)

Before running these:
* **paraphrase** — reword `DEFINITIONS_PARAPHRASE` in `run_grid_openai.py` (now a copy of the main definitions).
* **2- / 5-shot** — create `fewshot_examples.csv` (columns: `para_id, text`, the six labels with 1 = present, optional `evidence_<label>`). Example paragraphs that are also in the validation set are left out of that cell.
* **reverse** — nothing to prepare; codes are flipped back to 1 = present when saved.

In [ ]:
# all_pred = rg.run_grid(validation)                                 # all 9 cells
# all_pred = rg.run_grid(validation, cells=[("reverse", 0)])         # a single robustness cell